# Analyzing Historical Stock/Revenue Data and Building a Dashboard

This notebook downloads historical stock price data for Tesla (TSLA) and GameStop (GME), scrapes revenue data from the web, cleans it, and builds simple visualizations (price and revenue charts). It follows the instructions for the IBM hands-on lab assignment.

## Setup and imports

Run the following cell to import required libraries. If you're missing any library, install it using `pip install yfinance beautifulsoup4 requests matplotlib pandas`.

In [ ]:
# Imports
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import requests
import datetime
import numpy as np

plt.style.use('seaborn')
%matplotlib inline

## Helper functions
Functions to fetch stock data via `yfinance` and to scrape revenue tables from Macrotrends (or similar). The revenue-scraping function tries to be robust but websites change — if the scraping fails, adapt the target URL or CSS selectors.

In [ ]:
def get_stock_data(ticker, start_date=None, end_date=None):
    """Download historical stock data for ticker using yfinance."""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(start=start_date, end=end_date)
        hist.reset_index(inplace=True)
        return hist
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")
        return None

def get_revenue_data_macrotrends(company_path):
    """
    Scrapes historical revenue data from macrotrends.net pages.
    company_path: the path /stock-price/<company>/revenues or the full URL tail used by macrotrends.
    Example URL: https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue
    """
    try:
        # If user passed a full URL, use it; otherwise form the macrotrends URL
        if company_path.startswith('http'):
            url = company_path
        else:
            # try to build common macrotrends revenue url structure
            url = f"https://www.macrotrends.net/stocks/charts/{company_path}/revenue"

        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }
        page = requests.get(url, headers=headers, timeout=15)
        if page.status_code != 200:
            print('Failed to fetch', url, 'Status code:', page.status_code)
            return None

        soup = BeautifulSoup(page.content, 'html.parser')

        # Try to find the table by header text 'Revenue' or by table class
        # Macrotrends often has a table with 'historical data' and a class 'table'
        table = None
        # search for tables that have 'Date' and 'Revenue' headers
        for table_candidate in soup.find_all('table'):
            headers = [th.get_text(strip=True).lower() for th in table_candidate.find_all('th')]
            if 'date' in headers and any('revenue' in h for h in headers):
                table = table_candidate
                break

        if table is None:
            # fallback: find by class name
            table = soup.find('table', attrs={'class': 'historical_data_table'})

        if table is None:
            print('Revenue table not found on page; inspect the page manually. URL =', url)
            return None

        rows = []
        for tr in table.find_all('tr'):
            cols = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
            if len(cols) >= 2:
                rows.append(cols)

        # Build DataFrame, try to find columns 'Date' and 'Revenue'
        df = pd.DataFrame(rows)
        # If header row present, set first row as header
        first_row = df.iloc[0].str.lower().tolist()
        if 'date' in first_row and any('revenue' in c for c in first_row):
            df.columns = df.iloc[0]
            df = df[1:]
        # attempt to normalize columns to ['Date','Revenue']
        col_names = [c for c in df.columns]
        date_col = None
        revenue_col = None
        for c in col_names:
            c_low = str(c).lower()
            if 'date' in c_low:
                date_col = c
            if 'revenue' in c_low:
                revenue_col = c
        if date_col is None or revenue_col is None:
            # try first two cols
            date_col = df.columns[0]
            revenue_col = df.columns[1]

        rev_df = df[[date_col, revenue_col]].copy()
        rev_df.columns = ['Date', 'Revenue']

        # Clean revenue column: remove $ and commas; convert to numeric
        rev_df['Revenue'] = rev_df['Revenue'].str.replace(r'[^0-9.-]', '', regex=True)
        rev_df['Revenue'] = pd.to_numeric(rev_df['Revenue'], errors='coerce')

        # Parse dates
        def try_parse_date(s):
            for fmt in ('%Y-%m-%d', '%b %d, %Y', '%Y', '%m/%d/%Y'):
                try:
                    return datetime.datetime.strptime(s, fmt).date()
                except Exception:
                    continue
            try:
                return pd.to_datetime(s).date()
            except Exception:
                return None

        rev_df['Date'] = rev_df['Date'].astype(str).apply(lambda x: try_parse_date(x))
        rev_df = rev_df.dropna(subset=['Revenue'])
        rev_df = rev_df.sort_values('Date')
        rev_df = rev_df.reset_index(drop=True)
        return rev_df
    except Exception as e:
        print('Exception in get_revenue_data_macrotrends:', e)
        return None


## 1) Download Tesla (TSLA) historical stock data
We will download Tesla stock price (daily) using `yfinance` (from earliest available to today).

In [ ]:
start_date = '2000-01-01'
end_date = datetime.date.today().isoformat()
tesla_df = get_stock_data('TSLA', start_date=start_date, end_date=end_date)
print('Tesla data rows:', None if tesla_df is None else len(tesla_df))
tesla_df_head = tesla_df.head() if tesla_df is not None else None
tesla_df_tail = tesla_df.tail() if tesla_df is not None else None
tesla_df_head

### 1.1 Display the first five rows of the `tesla_df` dataframe (Question 1.2 in grading criteria)

In [ ]:
if tesla_df is not None:
    display(tesla_df.head())
else:
    print('Tesla data not available')

### 1.2 Display the last five rows of the `tesla_df` dataframe (Question 1.3 in grading criteria)

In [ ]:
if tesla_df is not None:
    display(tesla_df.tail())
else:
    print('Tesla data not available')

## 2) Download GameStop (GME) historical stock data

In [ ]:
gme_df = get_stock_data('GME', start_date=start_date, end_date=end_date)
print('GME data rows:', None if gme_df is None else len(gme_df))
if gme_df is not None:
    display(gme_df.head())
else:
    print('GME data not available')

### Display the last five rows of `gme_df`

In [ ]:
if gme_df is not None:
    display(gme_df.tail())
else:
    print('GME data not available')

## 3) Scrape Tesla revenue data from macrotrends (or a similar site)
We try Macrotrends' Tesla revenue page. If the page layout changes, the scraping function might need updating. If Macrotrends is blocked, consider downloading the CSV manually and loading it.

In [ ]:
tesla_revenue = get_revenue_data_macrotrends('TSLA/tesla')
if tesla_revenue is not None:
    print('Tesla revenue rows:', len(tesla_revenue))
    display(tesla_revenue.head())
else:
    print('Tesla revenue scraping failed; please check the site or URL.')

### Display the last five rows of tesla_revenue (Question 1.4 in grading criteria)

In [ ]:
if tesla_revenue is not None:
    display(tesla_revenue.tail())
else:
    print('Tesla revenue not available')

## 4) Scrape GameStop revenue data
Try GameStop revenue page. Update the company path if the page structure differs.

In [ ]:
gme_revenue = get_revenue_data_macrotrends('GME/gamestop')
if gme_revenue is not None:
    print('GME revenue rows:', len(gme_revenue))
    display(gme_revenue.head())
else:
    print('GME revenue scraping failed; please check the site or URL.')

### Display the last five rows of gme_revenue

In [ ]:
if gme_revenue is not None:
    display(gme_revenue.tail())
else:
    print('GME revenue not available')

## 5) Plots: Tesla stock (Close) and Tesla revenue
Plot the 'Close' column from the Tesla stock dataframe and the revenue data we scraped. This corresponds to Question 1.6 in the grading criteria.

In [ ]:
def plot_stock_and_revenue(stock_df, revenue_df, company_name='Company'):
    fig, ax = plt.subplots(2, 1, figsize=(12,10), sharex=False)
    if stock_df is not None:
        ax[0].plot(stock_df['Date'], stock_df['Close'])
        ax[0].set_title(f"{company_name} Stock Closing Price")
        ax[0].set_xlabel('Date')
        ax[0].set_ylabel('Close Price (USD)')
    else:
        ax[0].text(0.5, 0.5, 'Stock data not available', ha='center')

    if revenue_df is not None:
        ax[1].bar(revenue_df['Date'], revenue_df['Revenue'] / 1e6)
        ax[1].set_title(f"{company_name} Quarterly/Annual Revenue (Millions USD)")
        ax[1].set_xlabel('Date')
        ax[1].set_ylabel('Revenue (Millions USD)')
        fig.autofmt_xdate(rotation=45)
    else:
        ax[1].text(0.5, 0.5, 'Revenue data not available', ha='center')

    plt.tight_layout()
    plt.show()

plot_stock_and_revenue(tesla_df, tesla_revenue, company_name='Tesla')

## 6) Plots: GameStop stock (Close) and GameStop revenue
This corresponds to Question 1.7 in the grading criteria.

In [ ]:
plot_stock_and_revenue(gme_df, gme_revenue, company_name='GameStop')

## Additional questions (common IBM assignment asks)
Below are example snippets to answer some specific short questions often present in the grading rubric. Adjust column names/indices if your scraped tables have a different layout.

In [ ]:
# Example: Which parsers can be used in BeautifulSoup? (short textual answer)
parsers_answer = "BeautifulSoup supports several parsers including 'html.parser' (Python's built-in), 'lxml', and 'html5lib'."
print(parsers_answer)

# Example: Why convert 'Date' column to datetime? (short textual answer)
date_answer = (
    "Converting the 'Date' column to datetime enables time-series operations like sorting by date, resampling, plotting over time,"
    " and accurate indexing and filtering based on dates."
)
print(date_answer)


### Data cleaning example (remove $ and commas from price string column if required)
This snippet shows how to remove `$` symbols and commas from a price column and convert it to numeric (useful if you scraped price strings).

In [ ]:
def clean_price_series(series):
    # Remove $ and commas
    cleaned = series.astype(str).str.replace(r'[^0-9.-]', '', regex=True)
    return pd.to_numeric(cleaned, errors='coerce')

# Example usage (uncomment if you have a scraped dataframe with 'Price' column):
# df['Price_clean'] = clean_price_series(df['Price'])
# display(df.head())
print('Cleaning helper ready.')

## Save cleaned datasets (optional)
You can save the cleaned dataframes to CSV for later submission or inspection.

In [ ]:
if tesla_df is not None:
    tesla_df.to_csv('tesla_stock.csv', index=False)
if tesla_revenue is not None:
    tesla_revenue.to_csv('tesla_revenue.csv', index=False)
if gme_df is not None:
    gme_df.to_csv('gme_stock.csv', index=False)
if gme_revenue is not None:
    gme_revenue.to_csv('gme_revenue.csv', index=False)
print('Saved CSV files (if data available).')

## Notes & Troubleshooting
- If scraping fails due to site layout changes or access restrictions, consider downloading datasets manually (some Macrotrends pages provide CSV downloads) and loading them with `pd.read_csv()`.
- If `yfinance` download fails due to rate limits, try a smaller date range or use Google Colab which typically has fewer constraints.
- Make sure to provide the notebook file and any required CSVs when submitting to the IBM grader.

## End of Notebook
You can now run all cells and take screenshots required by the assignment rubric (first page screenshot, the individual output screenshots, etc.).